In [1]:
import pandas as pd
pd.set_option("display.width", 2000)
import matplotlib.pyplot as plt

In [2]:
# wczytanie komentarzy
pr_com_df = pd.read_parquet("../pr_comments.parquet")
hu_pr_com_df = pd.read_parquet("../human_pr_comments.parquet")

In [3]:
# wyciągnięcie url repozytoriów zwierające ludzkie PRy
pr_hu_df = pd.read_parquet("../human_pull_request.parquet")
human_urls = pd.DataFrame({"url": pr_hu_df["repo_url"].unique()})

In [4]:
# PRy AI pochodzące z repo, w których są ludzkie PR-y
pr_df = pd.read_parquet("../pull_request.parquet")
union_human_bot = pr_df.merge(human_urls, left_on="repo_url", right_on="url")

In [5]:
# wzbogacenie informacji
bot_pr_com = pr_com_df.merge(union_human_bot, left_on="pr_id", right_on="id", suffixes=(None, "_x"))

In [6]:
# zmiana typu kolumny pr_id z object na int64
hu_pr_com_df["pr_id"] = pd.to_numeric(hu_pr_com_df["pr_id"], errors="coerce").astype("Int64")
# wzbogacenie informacji
hu_pr_com = hu_pr_com_df.merge(pr_hu_df, left_on="pr_id", right_on="id", suffixes=(None, "_x"))

In [7]:
# Comments

In [8]:
# połączenie obu tabeli, aby ułatwić statystyki
bot_pr_com["source"] = "AI"
hu_pr_com["source"] = "human"

comments = pd.concat([bot_pr_com, hu_pr_com], ignore_index=True)

In [9]:
# podstawowe statystyki
summary = comments.groupby(["source", "user_type"]).agg(
    total_comments=("id", "count"),
    unique_commenters=("user", "nunique"),
    unique_prs=("pr_id", "nunique")
).reset_index()

summary

,source,user_type,total_comments,unique_commenters,unique_prs
0,AI,Bot,16659,75,6242
1,AI,User,6799,1182,3143
2,human,Bot,6378,89,3137
3,human,User,5262,1478,2231


In [10]:
# statystyki komentarzy ogólne
comments_per_pr = (
    comments.groupby(["source", "user_type", "pr_id"])
    .size()
    .reset_index(name="comments_count")
)

comments_per_pr.groupby(["source", "user_type"])["comments_count"].describe()

count      mean       std  min  25%  50%  75%   max
source user_type                                                      
AI     Bot        6242.0  2.668856  2.163873  1.0  1.0  2.0  3.0  30.0
       User       3143.0  2.163220  2.408826  1.0  1.0  1.0  2.0  28.0
human  Bot        3137.0  2.033153  1.626133  1.0  1.0  1.0  3.0  22.0
       User       2231.0  2.358584  3.077557  1.0  1.0  1.0  2.0  30.0

In [11]:
# statystyki długości komentarzy 
comments["body_length"] = comments["body"].fillna("").apply(len)
comments.groupby(["source", "user_type"])["body_length"].describe()

count         mean          std  min    25%    50%     75%       max
source user_type                                                                        
AI     Bot        16659.0  1969.294916  7160.064572  7.0  378.0  581.0  1340.5  147598.0
       User        6799.0   684.619797  3504.851206  1.0   41.0  112.0   334.0  203802.0
human  Bot         6378.0  3312.063500  9723.612430  9.0  380.0  978.0  2464.0  256884.0
       User        5262.0   712.835424  2879.568095  1.0   63.0  165.0   504.0  103223.0

In [12]:
# jaki rozkład recenzentów (ogólny)
comments_mix = (
    comments.groupby(["source", "pr_id"])["user_type"]
    .unique()
    .reset_index()
)

def classify_mix(user_types):
    if len(user_types) == 1:
        if user_types[0] == "Bot":
            return "bot-only"
        else:
            return "human-only"
    else:
        return "mixed"

comments_mix["comments_mix_type"] = comments_mix["user_type"].apply(classify_mix)

mix_summary = (
    comments_mix.groupby(["source", "comments_mix_type"])
    .size()
    .reset_index(name="count")
)

mix_summary["percent"] = mix_summary.groupby("source")["count"].transform(
    lambda x: 100 * x / x.sum()
)

mix_summary

,source,comments_mix_type,count,percent
0,AI,bot-only,3814,54.822481
1,AI,human-only,715,10.277418
2,AI,mixed,2428,34.900101
3,human,bot-only,2085,48.308619
4,human,human-only,1179,27.316960
5,human,mixed,1052,24.374421


In [13]:
# jaki rozkład recenzentów (szczegółowy) UWAGA: wolno działa

comments_mix_v2 = (
    comments.groupby(["source", "pr_id"]).agg({
        "user_type": lambda x: list(x.unique()),
        'id': "count"
    })
    .reset_index()
    .rename(columns={"id": "comment_count"})
)

def classify_mix_v2(row):
    user_types = row["user_type"]
    count = row["comment_count"]
    
    if len(user_types) == 1:
        if user_types[0] == "Bot":
            return f"bot-only ({count})"
        else:
            return f"human-only ({count})"
    else:
        pr_rows = comments[(comments["source"] == row["source"]) & (comments["pr_id"] == row["pr_id"])]
        bot_count = (pr_rows["user_type"] == "Bot").sum()
        human_count = (pr_rows["user_type"] != "Bot").sum()
        return f"mixed ({count}) (AI: {bot_count}, H: {human_count})"

comments_mix_v2["comment_mix_detailed"] = comments_mix_v2.apply(classify_mix_v2, axis=1)

mix_detailed_summary = (
    comments_mix_v2.groupby(["source", "comment_mix_detailed"])
    .size()
    .reset_index(name="count")
)

mix_detailed_summary["percent"] = mix_detailed_summary.groupby("source")["count"].transform(
    lambda x: 100 * x / x.sum()
)

pd.set_option("display.max_rows", None)
mix_detailed_summary.sort_values(["source", "percent"], ascending=[True, False], inplace=True)

mix_detailed_summary

,source,comment_mix_detailed,count,percent
0,AI,bot-only (1),1342,19.289924
6,AI,bot-only (2),1004,14.431508
7,AI,bot-only (3),661,9.501222
100,AI,"mixed (2) (AI: 1, H: 1)",536,7.704470
15,AI,human-only (1),511,7.345120
121,AI,"mixed (3) (AI: 2, H: 1)",431,6.195199
9,AI,bot-only (4),416,5.979589
10,AI,bot-only (5),179,2.572948
120,AI,"mixed (3) (AI: 1, H: 2)",148,2.127354
132,AI,"mixed (4) (AI: 2, H: 2)",147,2.112980


In [14]:
# najczęściej występujące słowa w recenzjach
from collections import Counter
import re
#import nltk
#from nltk.corpus import stopwords

stop_words = {
    "the", "a", "an", "and", "or", "is", "it", "to", "in", "of", "for", "on", "this",
    "that", "with", "as", "by", "from", "are", "was", "be", "at", "which", "have",
    "has", "had", "will", "can", "not", "if", "else", "when", "then", "about", "up",
    "out", "so", "no", "yes", "your", "our", "their", "its", "but", "we", "you", "they"
}

def analyze_common(df, source, user_type, top, isState, state = None):
    if isState:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type) & (df["state"] == state)] 
    else:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type)] 
        
    all_text = " ".join(subset["body"].dropna().astype(str)).lower()
    words = re.findall(r"\b\w+\b", all_text)
    words = [w for w in words if w not in stop_words and len(w) > 2]
    word_count = Counter(words)

    return word_count.most_common(top)
    
for src in comments["source"].unique():
    for utype in comments["user_type"].unique():
        if utype != None:
            top_words = analyze_common(comments, src, utype, 20, False)
            
            print(f"Source: {src}, User type: {utype}")
            for word, count in top_words:
                print(f"{word}: {count}")
            print("==============================")

Source: AI, User type: User
test: 14483
bun: 11093
https: 9409
com: 8143
code: 8099
href: 5740
github: 5274
src: 3558
builds: 3379
error: 3268
blob: 3158
buildkite: 2997
oven: 2977
tests: 2812
node: 2613
build: 2202
x64: 2051
parallel: 2037
file: 1966
Source: AI, User type: Bot
https: 44995
build: 37826
pyc: 26082
summary: 24870
coderabbit: 23225
coderabbitai: 22706
com: 22046
details: 20569
github: 19414
comment: 19333
package: 17943
review: 17832
comments: 17627
index: 16511
vercel: 16183
add: 15832
app: 15268
assets: 15074
map: 14970
nbsp: 14476
Source: human, User type: User
https: 5353
test: 3957
com: 3862
sub: 3011
code: 2843
github: 2470
bun: 2401
tests: 2106
changes: 1457
documentation: 1361
align: 1347
file: 1330
cla: 1323
error: 1323
pull: 1320
href: 1301
src: 1264
right: 1218
python: 1214
__webpack_require__: 1206
Source: human, User type: Bot
https: 32510
summary: 18060
com: 17734
github: 16171
details: 15344
build: 13589
nbsp: 13501
coderabbit: 13342
coderabbitai: 12781
ap

In [15]:
from collections import Counter
import re
import matplotlib.pyplot as plt
import pandas as pd
import csv
import os

# ===== STOP WORDS =====
stop_words = {
    "the", "a", "an", "and", "or", "is", "it", "to", "in", "of", "for", "on", "this",
    "that", "with", "as", "by", "from", "are", "was", "be", "at", "which", "have",
    "has", "had", "will", "can", "not", "if", "else", "when", "then", "about", "up",
    "out", "so", "no", "yes", "your", "our", "their", "its", "but", "we", "you", "they"
}

# ===== ANALIZA =====
def analyze_common(df, source, user_type, top=20, isState=False, state=None):
    if isState:
        subset = df[
            (df["source"] == source)
            & (df["user_type"] == user_type)
            & (df["state"] == state)
        ]
    else:
        subset = df[
            (df["source"] == source)
            & (df["user_type"] == user_type)
        ]

    all_text = " ".join(subset["body"].dropna().astype(str)).lower()
    words = re.findall(r"\b\w+\b", all_text)
    words = [w for w in words if w not in stop_words and len(w) > 2]
    word_count = Counter(words)

    return word_count.most_common(top)

def plot_bar(top_words, title, save_path):
    words = [w for w, _ in top_words]
    freqs = [f for _, f in top_words]

    plt.figure(figsize=(10, 6))
    plt.barh(words[::-1], freqs[::-1], color="cornflowerblue")
    plt.title(title, fontsize=14)
    plt.xlabel("Frequency")
    plt.ylabel("Word")
    plt.tight_layout()

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


output_dir_csv = "csv/comments_body_without_state"
output_dir_plot = "plots/comments_body_without_state"
os.makedirs(output_dir_csv, exist_ok=True)
os.makedirs(output_dir_plot, exist_ok=True)

for src in comments["source"].unique():
    for utype in comments["user_type"].unique():
        if utype is None:
            continue

        top_words = analyze_common(comments, src, utype, top=20, isState=False)
        if len(top_words) == 0:
            continue

        title = f"Source: {src}, User type: {utype}"

        # Zapis wykresu
        plot_path = os.path.join(output_dir_plot, f"{src}_{utype}.png")
        plot_bar(top_words, title, plot_path)

        # Zapis CSV w formacie: "weight","word","color"
        df_export = pd.DataFrame(
            [(freq, word, "", "") for word, freq in top_words],
            columns=["weight", "word", "color", "url"]
        )
        csv_path = os.path.join(output_dir_csv, f"{src}_{utype}.csv")
        df_export.to_csv(csv_path, index=False, quoting=csv.QUOTE_ALL)

        print(f"Zapisano: {csv_path}")
        print(f"Zapisano wykres: {plot_path}")


Zapisano: csv/comments_body_without_state/AI_User.csv
Zapisano wykres: plots/comments_body_without_state/AI_User.png
Zapisano: csv/comments_body_without_state/AI_Bot.csv
Zapisano wykres: plots/comments_body_without_state/AI_Bot.png
Zapisano: csv/comments_body_without_state/human_User.csv
Zapisano wykres: plots/comments_body_without_state/human_User.png
Zapisano: csv/comments_body_without_state/human_Bot.csv
Zapisano wykres: plots/comments_body_without_state/human_Bot.png
